# Inference And Create Submission

Notebook n?y kh?ng train. N? ??c model ?? c? trong `outputs`, render ?nh theo `test_poses.csv`, validate output v? t?o `submission.zip` ?? n?p Kaggle.

Notebook t?i artifact t? kernel train, t? ??ng ph?t hi?n model ? 30k, render theo `test_poses.csv`, validate v? t?o `submission.zip`.


In [ ]:
from pathlib import Path
import os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

DATASET_ROOT = Path('/kaggle/input/datasets/dangthtai/bts-digital-twin-dataset')
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path('VAI_NVS_DATA')
WORK_DIR = Path('/kaggle/working')
if not WORK_DIR.exists():
    WORK_DIR = Path.cwd()
REPO_DIR = WORK_DIR / 'gaussian-splatting'
if (WORK_DIR / 'train.py').exists():
    REPO_DIR = WORK_DIR
OUTPUT_ROOT = WORK_DIR / 'outputs'  # Đường dẫn tới thư mục chứa outputs (Có thể thay bằng Path('/kaggle/input/.../outputs') nếu dùng Kaggle Input)
SUBMISSION_DIR = WORK_DIR / 'submission'
SUBMISSION_ZIP = WORK_DIR / 'submission.zip'

GIT_REPO_URL = 'https://github.com/graphdeco-inria/gaussian-splatting.git'

SUBMISSION_SPLITS = None  # None: infer split(s) from downloaded outputs
ONLY_SCENES = None  # None: infer scene(s) from downloaded outputs
ITERATIONS = 30000

# File name in submission follows image_name from CSV. Use 'stem_png' only if the competition requires PNG stems.
SUBMISSION_IMAGE_NAME_MODE = 'csv_exact'  # 'csv_exact' or 'stem_png'

# Maximum experimental render quality. This renders all Gaussians per pose and may OOM on Kaggle T4.
RENDER_USE_ANTIALIASING = True
RENDER_MAX_GAUSSIAN_SCALE = 1000000000.0
RENDER_MAX_RENDER_POINTS = 0
RENDER_MAX_VIEW_POINTS = 0
RENDER_VIEW_CULL_MARGIN = 0.0
RENDER_MIN_VIEW_DEPTH = 0.0
RENDER_MAX_SCREEN_RADIUS_PX = 0.0
RENDER_RESOLUTION_SCALE = 1.0
RENDER_CONVERT_SHS_PYTHON = True
RENDER_COMPUTE_COV3D_PYTHON = True

print('Dataset root:', DATASET_ROOT)
print('Submission splits:', SUBMISSION_SPLITS or 'AUTO')
print('Only scenes:', ONLY_SCENES or 'AUTO')
print('Iterations:', ITERATIONS)
print('Render safety:', 'AA=', RENDER_USE_ANTIALIASING, 'max_scale=', RENDER_MAX_GAUSSIAN_SCALE, 'max_render_points=', RENDER_MAX_RENDER_POINTS, 'max_view_points=', RENDER_MAX_VIEW_POINTS, 'view_margin=', RENDER_VIEW_CULL_MARGIN, 'min_depth=', RENDER_MIN_VIEW_DEPTH, 'max_screen_radius_px=', RENDER_MAX_SCREEN_RADIUS_PX, 'resolution_scale=', RENDER_RESOLUTION_SCALE, 'convert_SHs_python=', RENDER_CONVERT_SHS_PYTHON, 'compute_cov3D_python=', RENDER_COMPUTE_COV3D_PYTHON)
print('Output root:', OUTPUT_ROOT)
print('Submission dir:', SUBMISSION_DIR)
print('Submission zip:', SUBMISSION_ZIP)
print('CUDA alloc conf:', os.environ.get('PYTORCH_CUDA_ALLOC_CONF'))


In [ ]:
import shutil
import subprocess
from pathlib import Path

marker_pattern = f'*/*/point_cloud/iteration_{ITERATIONS}/point_cloud.ply'
model_markers = sorted(OUTPUT_ROOT.glob(marker_pattern)) if OUTPUT_ROOT.exists() else []

if not model_markers:
    raise FileNotFoundError(
        f'Khong tim thay mo hinh tai {OUTPUT_ROOT}. \n'
        f'Vui long kiem tra lai duong dan OUTPUT_ROOT trong cell cau hinh.'
    )

print('Trained models found:')
for marker in model_markers:
    print(' -', marker)

!nvidia-smi

In [ ]:
import os, shutil, subprocess

if not REPO_DIR.exists():
    subprocess.check_call(['git', 'clone', GIT_REPO_URL, str(REPO_DIR)])
elif not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repository')
else:
    print('Repo exists:', REPO_DIR)

def init_required_submodule(path, fallback_url=None, recursive=False):
    target = REPO_DIR / path
    command = ['git', 'submodule', 'update', '--init']
    if recursive:
        command.append('--recursive')
    command.append(path)
    try:
        subprocess.check_call(command, cwd=REPO_DIR)
    except subprocess.CalledProcessError:
        if fallback_url is None:
            raise
        print(f'Submodule {path} failed; cloning fallback {fallback_url}')
        shutil.rmtree(target, ignore_errors=True)
        target.parent.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(['git', 'clone', fallback_url, str(target)])

init_required_submodule('submodules/diff-gaussian-rasterization', recursive=True)
init_required_submodule('submodules/fused-ssim')
init_required_submodule('submodules/simple-knn', 'https://github.com/camenduru/simple-knn.git')
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())


In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

!python -m pip install --upgrade "pip<27" "setuptools<82" wheel ninja
!python -m pip install plyfile tqdm opencv-python joblib pillow

import torch
print('python:', sys.version)
print('torch:', torch.__version__)
print('torch cuda:', torch.version.cuda)
print('cuda available:', torch.cuda.is_available())
print('nvcc:', shutil.which('nvcc'))

if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError(
        'Kaggle runtime is using CPU-only PyTorch. Enable the GPU accelerator, '
        'sau do Runtime -> Restart runtime va chay lai notebook tu dau.'
    )

if shutil.which('nvcc') is None:
    raise RuntimeError(
        'Khong tim thay nvcc CUDA compiler. 3DGS can nvcc de build CUDA extensions. '
        'Enable the Kaggle GPU accelerator, restart the session, then run again.'
    )

subprocess.run(['nvcc', '--version'], check=False)

# Quality-first install: use the accelerated rasterizer branch required by --optimizer_type sparse_adam.
# Without this branch, train.py prints: "Trying to use sparse adam but it is not installed".
diff_rast = Path('submodules/diff-gaussian-rasterization')
subprocess.run(['git', 'fetch', 'origin', '3dgs_accel'], cwd=diff_rast, check=True)
subprocess.run(['git', 'checkout', '3dgs_accel'], cwd=diff_rast, check=True)
shutil.rmtree(diff_rast / 'build', ignore_errors=True)

!python -m pip uninstall -y diff-gaussian-rasterization
!MAX_JOBS=2 FORCE_CUDA=1 python -m pip install --no-build-isolation ./submodules/diff-gaussian-rasterization
!MAX_JOBS=2 FORCE_CUDA=1 python -m pip install --no-build-isolation ./submodules/simple-knn
!MAX_JOBS=2 FORCE_CUDA=1 python -m pip install --no-build-isolation ./submodules/fused-ssim

import diff_gaussian_rasterization
from diff_gaussian_rasterization import SparseGaussianAdam
import simple_knn._C
import fused_ssim
print('CUDA extensions import OK')
print('SparseGaussianAdam import OK:', SparseGaussianAdam)


In [ ]:
import csv
from pathlib import Path


def find_challenge_scenes(root, splits):
    root = Path(root)
    scenes = []
    for split in splits:
        for test_csv in sorted(root.rglob(f'phase1/{split}/*/test/test_poses.csv')):
            scene_dir = test_csv.parents[1]
            train_dir = scene_dir / 'train'
            if (train_dir / 'images').exists() and (train_dir / 'sparse' / '0' / 'cameras.bin').exists():
                scenes.append({
                    'split': split,
                    'scene_name': scene_dir.name,
                    'scene_dir': scene_dir,
                    'train_dir': train_dir,
                    'test_csv': test_csv,
                })
    return scenes


def validate_submission_scene(scene):
    test_csv = Path(scene['test_csv'])
    with test_csv.open(newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    required_cols = {'image_name', 'qw', 'qx', 'qy', 'qz', 'tx', 'ty', 'tz', 'fx', 'fy', 'cx', 'cy', 'width', 'height'}
    missing_cols = required_cols - set(rows[0].keys() if rows else [])
    if missing_cols:
        raise RuntimeError(f"{scene['scene_name']}: test_poses.csv missing columns: {sorted(missing_cols)}")
    marker = OUTPUT_ROOT / scene['split'] / scene['scene_name'] / 'point_cloud' / f'iteration_{ITERATIONS}' / 'point_cloud.ply'
    if not marker.exists():
        raise FileNotFoundError(f'Expected trained model not found: {marker}')
    print(f"Check {scene['split']}/{scene['scene_name']}: test_poses={len(rows)}, model={marker}")


trained_keys = {(p.parents[3].name, p.parents[2].name) for p in model_markers}
trained_splits = sorted({split for split, _ in trained_keys})
search_splits = SUBMISSION_SPLITS or trained_splits
submission_scenes = find_challenge_scenes(DATASET_ROOT, search_splits)
submission_scenes = [s for s in submission_scenes if (s['split'], s['scene_name']) in trained_keys]
if ONLY_SCENES is not None:
    submission_scenes = [s for s in submission_scenes if s['scene_name'] in ONLY_SCENES]
found_keys = {(s['split'], s['scene_name']) for s in submission_scenes}
missing_dataset_scenes = sorted(trained_keys - found_keys)
if missing_dataset_scenes:
    raise RuntimeError(f'Trained model(s) have no matching dataset test_poses.csv: {missing_dataset_scenes}')
if not submission_scenes:
    raise RuntimeError(f'No scene can be inferred from models in {OUTPUT_ROOT}')
for scene in submission_scenes:
    validate_submission_scene(scene)
print('Submission scenes:', [(s['split'], s['scene_name']) for s in submission_scenes])


In [ ]:
render_script = r'''
import argparse
import csv
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import torch
import torchvision

from gaussian_renderer import render
from scene.gaussian_model import GaussianModel
from utils.graphics_utils import focal2fov, getProjectionMatrix, getWorld2View2
from utils.system_utils import searchForMaxIteration


def qvec2rotmat(qvec):
    qvec = np.asarray(qvec, dtype=np.float64)
    norm = np.linalg.norm(qvec)
    if not np.isfinite(norm) or norm <= 0:
        raise ValueError(f"Invalid quaternion: {qvec.tolist()}")
    qvec = qvec / norm
    return np.array(
        [
            [1 - 2 * qvec[2] ** 2 - 2 * qvec[3] ** 2, 2 * qvec[1] * qvec[2] - 2 * qvec[0] * qvec[3], 2 * qvec[3] * qvec[1] + 2 * qvec[0] * qvec[2]],
            [2 * qvec[1] * qvec[2] + 2 * qvec[0] * qvec[3], 1 - 2 * qvec[1] ** 2 - 2 * qvec[3] ** 2, 2 * qvec[2] * qvec[3] - 2 * qvec[0] * qvec[1]],
            [2 * qvec[3] * qvec[1] - 2 * qvec[0] * qvec[2], 2 * qvec[2] * qvec[3] + 2 * qvec[0] * qvec[1], 1 - 2 * qvec[1] ** 2 - 2 * qvec[2] ** 2],
        ]
    )


class CsvCamera:
    def __init__(self, row, uid, render_scale=1.0):
        self.uid = uid
        self.image_name = row["image_name"]
        self.original_width = int(float(row["width"]))
        self.original_height = int(float(row["height"]))
        if render_scale <= 0 or render_scale > 1:
            raise ValueError(f"render_scale must be in (0, 1], got {render_scale}")
        self.image_width = max(1, int(round(self.original_width * render_scale)))
        self.image_height = max(1, int(round(self.original_height * render_scale)))
        fx, fy = float(row["fx"]), float(row["fy"])
        self.fx = fx
        self.fy = fy
        if self.original_width <= 0 or self.original_height <= 0:
            raise ValueError(f"{self.image_name}: invalid image size {self.original_width}x{self.original_height}")
        if not np.isfinite([fx, fy]).all() or fx <= 0 or fy <= 0:
            raise ValueError(f"{self.image_name}: invalid focal lengths fx={fx}, fy={fy}")
        self.FoVx = focal2fov(fx, self.original_width)
        self.FoVy = focal2fov(fy, self.original_height)
        if not np.isfinite([self.FoVx, self.FoVy]).all():
            raise ValueError(f"{self.image_name}: invalid FoV values FoVx={self.FoVx}, FoVy={self.FoVy}")

        qvec = [float(row["qw"]), float(row["qx"]), float(row["qy"]), float(row["qz"])]
        rotation = qvec2rotmat(qvec).T
        translation = np.array([float(row["tx"]), float(row["ty"]), float(row["tz"])])
        if not np.isfinite(translation).all():
            raise ValueError(f"{self.image_name}: invalid translation {translation.tolist()}")

        self.world_view_transform = torch.tensor(
            getWorld2View2(rotation, translation, np.array([0.0, 0.0, 0.0]), 1.0)
        ).transpose(0, 1).cuda()
        self.projection_matrix = getProjectionMatrix(
            znear=0.01, zfar=100.0, fovX=self.FoVx, fovY=self.FoVy
        ).transpose(0, 1).cuda()
        self.full_proj_transform = self.world_view_transform.unsqueeze(0).bmm(
            self.projection_matrix.unsqueeze(0)
        ).squeeze(0)
        self.camera_center = self.world_view_transform.inverse()[3, :3]


def output_name(image_name, mode):
    filename = Path(image_name).name
    if mode == "stem_png":
        return Path(filename).stem + ".png"
    return filename


def _filter_gaussians(gaussians, valid, reason):
    kept = int(valid.sum().item())
    total = int(valid.numel())
    removed = total - kept
    if removed == 0:
        print(f"{reason}: kept all {total} points")
        return
    if kept == 0:
        raise RuntimeError(f"{reason} removed every point; trained point cloud is invalid.")

    gaussians._xyz = torch.nn.Parameter(gaussians._xyz[valid].contiguous().requires_grad_(True))
    gaussians._features_dc = torch.nn.Parameter(gaussians._features_dc[valid].contiguous().requires_grad_(True))
    gaussians._features_rest = torch.nn.Parameter(gaussians._features_rest[valid].contiguous().requires_grad_(True))
    gaussians._opacity = torch.nn.Parameter(gaussians._opacity[valid].contiguous().requires_grad_(True))
    gaussians._scaling = torch.nn.Parameter(gaussians._scaling[valid].contiguous().requires_grad_(True))
    gaussians._rotation = torch.nn.Parameter(gaussians._rotation[valid].contiguous().requires_grad_(True))
    print(f"{reason}: removed {removed}, kept {kept}")


def slice_gaussians(gaussians, mask):
    subset = GaussianModel(gaussians.max_sh_degree)
    subset.active_sh_degree = gaussians.active_sh_degree
    subset._xyz = torch.nn.Parameter(gaussians._xyz[mask].contiguous().requires_grad_(True))
    subset._features_dc = torch.nn.Parameter(gaussians._features_dc[mask].contiguous().requires_grad_(True))
    subset._features_rest = torch.nn.Parameter(gaussians._features_rest[mask].contiguous().requires_grad_(True))
    subset._opacity = torch.nn.Parameter(gaussians._opacity[mask].contiguous().requires_grad_(True))
    subset._scaling = torch.nn.Parameter(gaussians._scaling[mask].contiguous().requires_grad_(True))
    subset._rotation = torch.nn.Parameter(gaussians._rotation[mask].contiguous().requires_grad_(True))
    return subset


def view_cull_mask(gaussians, camera, margin, min_depth, max_screen_radius_px, chunk_size=1000000):
    xyz = gaussians.get_xyz
    keep = torch.zeros(xyz.shape[0], dtype=torch.bool, device=xyz.device)
    transform = camera.full_proj_transform
    view_transform = camera.world_view_transform
    max_scale = gaussians.get_scaling.max(dim=1).values
    focal = max(float(camera.fx), float(camera.fy))
    for start in range(0, xyz.shape[0], chunk_size):
        end = min(start + chunk_size, xyz.shape[0])
        points = xyz[start:end]
        ones = torch.ones((points.shape[0], 1), dtype=points.dtype, device=points.device)
        points_hom = torch.cat((points, ones), dim=1)
        view = points_hom @ view_transform
        depth = view[:, 2]
        clip = points_hom @ transform
        w = clip[:, 3]
        ndc = clip[:, :3] / (w[:, None] + 1e-7)
        screen_radius = max_scale[start:end] * focal / torch.clamp(depth.abs(), min=1e-6)
        keep[start:end] = (
            torch.isfinite(ndc).all(dim=1)
            & torch.isfinite(w)
            & torch.isfinite(depth)
            & torch.isfinite(screen_radius)
            & (w > 0)
            & (depth.abs() >= min_depth)
            & ((max_screen_radius_px <= 0) | (screen_radius <= max_screen_radius_px))
            & (ndc[:, 0].abs() <= margin)
            & (ndc[:, 1].abs() <= margin)
            & (ndc[:, 2] >= -0.25)
            & (ndc[:, 2] <= 1.25)
        )
    return keep


def sanitize_gaussians(gaussians, max_scale, max_points):
    with torch.no_grad():
        scale = gaussians.get_scaling
        opacity = gaussians.get_opacity.squeeze(-1)
        rotation_norm = torch.linalg.norm(gaussians._rotation, dim=1)
        max_scale_per_point = scale.max(dim=1).values
        valid = (
            torch.isfinite(gaussians._xyz).all(dim=1)
            & torch.isfinite(gaussians._features_dc).flatten(start_dim=1).all(dim=1)
            & torch.isfinite(gaussians._features_rest).flatten(start_dim=1).all(dim=1)
            & torch.isfinite(gaussians._opacity).all(dim=1)
            & torch.isfinite(opacity)
            & torch.isfinite(gaussians._scaling).all(dim=1)
            & torch.isfinite(scale).all(dim=1)
            & (max_scale_per_point <= max_scale)
            & torch.isfinite(gaussians._rotation).all(dim=1)
            & torch.isfinite(rotation_norm)
            & (rotation_norm > 1e-8)
        )
        print(
            "Gaussian stats:",
            f"count={valid.numel()}",
            f"scale_max={float(max_scale_per_point.max().item()):.6g}",
            f"scale_p99={float(torch.quantile(max_scale_per_point.float(), 0.99).item()):.6g}",
            f"opacity_max={float(opacity.max().item()):.6g}",
            f"opacity_p50={float(torch.quantile(opacity.float(), 0.50).item()):.6g}",
        )
        _filter_gaussians(gaussians, valid, "Gaussian sanity check")

        if max_points > 0 and gaussians.get_xyz.shape[0] > max_points:
            opacity = gaussians.get_opacity.squeeze(-1)
            scale = gaussians.get_scaling
            max_scale_per_point = scale.max(dim=1).values
            score = opacity / (1.0 + max_scale_per_point)
            threshold = torch.topk(score, k=max_points, largest=True, sorted=False).values.min()
            keep = score >= threshold
            if int(keep.sum().item()) > max_points:
                selected = torch.topk(score, k=max_points, largest=True, sorted=False).indices
                keep = torch.zeros_like(score, dtype=torch.bool)
                keep[selected] = True
            _filter_gaussians(gaussians, keep, f"Gaussian render budget max_points={max_points}")


def main():
    parser = argparse.ArgumentParser(description="Render challenge CSV camera poses with a trained 3DGS model.")
    parser.add_argument("-m", "--model_path", required=True)
    parser.add_argument("--test_csv", required=True)
    parser.add_argument("--out_dir", required=True)
    parser.add_argument("--iteration", type=int, default=-1)
    parser.add_argument("--sh_degree", type=int, default=3)
    parser.add_argument("--white_background", action="store_true")
    parser.add_argument("--antialiasing", action="store_true")
    parser.add_argument("--output_name_mode", default="csv_exact", choices=["csv_exact", "stem_png"])
    parser.add_argument("--max_gaussian_scale", type=float, default=1.0)
    parser.add_argument("--max_render_points", type=int, default=0)
    parser.add_argument("--view_cull_margin", type=float, default=1.25)
    parser.add_argument("--max_view_points", type=int, default=800000)
    parser.add_argument("--min_view_depth", type=float, default=0.05)
    parser.add_argument("--max_screen_radius_px", type=float, default=180.0)
    parser.add_argument("--render_scale", type=float, default=1.0)
    parser.add_argument("--convert_SHs_python", action="store_true")
    parser.add_argument("--compute_cov3D_python", action="store_true")
    parser.add_argument("--debug_cuda_sync", action="store_true")
    args = parser.parse_args()

    model_path = Path(args.model_path)
    iteration = args.iteration if args.iteration != -1 else searchForMaxIteration(str(model_path / "point_cloud"))
    ply_path = model_path / "point_cloud" / f"iteration_{iteration}" / "point_cloud.ply"
    if not ply_path.exists():
        raise FileNotFoundError(ply_path)

    gaussians = GaussianModel(args.sh_degree)
    gaussians.load_ply(str(ply_path))
    sanitize_gaussians(gaussians, args.max_gaussian_scale, args.max_render_points)
    torch.cuda.empty_cache()
    pipeline = SimpleNamespace(convert_SHs_python=args.convert_SHs_python, compute_cov3D_python=args.compute_cov3D_python, debug=False, antialiasing=args.antialiasing)
    background = torch.tensor([1, 1, 1] if args.white_background else [0, 0, 0], dtype=torch.float32, device="cuda")

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    with open(args.test_csv, newline="", encoding="utf-8") as csv_file:
        rows = list(csv.DictReader(csv_file))

    print("Rendering", len(rows), "poses to", out_dir)
    with torch.no_grad():
        for idx, row in enumerate(rows):
            try:
                camera = CsvCamera(row, idx, args.render_scale)
                render_gaussians = gaussians
                if args.view_cull_margin > 0:
                    view_mask = view_cull_mask(gaussians, camera, args.view_cull_margin, args.min_view_depth, args.max_screen_radius_px)
                    view_count = int(view_mask.sum().item())
                    if view_count == 0:
                        raise RuntimeError(f"View culling removed every point for pose {idx}")
                    if args.max_view_points > 0 and view_count > args.max_view_points:
                        view_indices = torch.nonzero(view_mask, as_tuple=False).squeeze(1)
                        opacity = gaussians.get_opacity.squeeze(-1)[view_indices]
                        selected = view_indices[torch.topk(opacity, k=args.max_view_points, largest=True, sorted=False).indices]
                        view_mask = torch.zeros_like(view_mask)
                        view_mask[selected] = True
                        view_count = int(view_mask.sum().item())
                    print(f"Pose {idx}: view culling kept {view_count}/{gaussians.get_xyz.shape[0]} points")
                    render_gaussians = slice_gaussians(gaussians, view_mask)
                    torch.cuda.empty_cache()

                image = render(camera, render_gaussians, pipeline, background)["render"]
                if args.render_scale != 1.0:
                    image = torch.nn.functional.interpolate(
                        image.unsqueeze(0),
                        size=(camera.original_height, camera.original_width),
                        mode="bilinear",
                        align_corners=False,
                    ).squeeze(0).clamp(0, 1)
                if args.debug_cuda_sync:
                    torch.cuda.synchronize()
                path = out_dir / output_name(row["image_name"], args.output_name_mode)
                torchvision.utils.save_image(image, str(path))
            except Exception as exc:
                print(f"Failed at pose {idx}: {row}")
                raise
            if (idx + 1) % 25 == 0:
                print(f"  {idx + 1}/{len(rows)}")


if __name__ == "__main__":
    main()
'''

(REPO_DIR / 'render_test_poses.py').write_text(render_script, encoding='utf-8')
print('Wrote:', REPO_DIR / 'render_test_poses.py')

In [ ]:
import os, subprocess
from pathlib import Path

def print_resource_snapshot(label):
    print('\nResource snapshot:', label)
    subprocess.run(['df', '-h', str(WORK_DIR)], check=False)
    subprocess.run(['free', '-h'], check=False)
    subprocess.run(['nvidia-smi'], check=False)

def run_stream(cmd, cwd, log_path, env=None):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    child_env = os.environ.copy()
    child_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    if env:
        child_env.update(env)
    print('Running:', ' '.join(map(str, cmd)))
    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(cmd, cwd=cwd, env=child_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
        ret = proc.wait()
    if ret != 0:
        lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print('\n'.join(lines[-120:]))
        raise RuntimeError(f'Command failed with exit code {ret}. Log: {log_path}')


## Compare metrics at 15k and 30k

Public scenes include ground-truth files in `test/images`. The cells below render `test_poses.csv` at both checkpoints and compare each result with the public test image having the same `image_name`. Private scenes without `test/images` are skipped because their metrics cannot be calculated offline.


In [ ]:
# Checkpoints to compare. Both point_cloud folders must exist in every selected model.
METRIC_ITERATIONS = [ITERATIONS]  # kaggle_train.ipynb only saves a point cloud at 30k
RUN_CHECKPOINT_METRICS = True
METRIC_INCLUDE_LPIPS = True  # Set False for a faster PSNR/SSIM-only comparison.

print('Metric iterations:', METRIC_ITERATIONS)
print('Metric scenes:', [(s['split'], s['scene_name']) for s in submission_scenes])


In [ ]:
import json
import math
import pandas as pd
import torch
import torchvision.transforms.functional as TF
from PIL import Image


def evaluate_render_folder(render_dir, gt_dir, include_lpips=True):
    # Import repository implementations so the numbers match metrics.py.
    from utils.image_utils import psnr
    from utils.loss_utils import ssim
    lpips_model = None
    if include_lpips:
        from lpipsPyTorch.modules.lpips import LPIPS
        lpips_model = LPIPS(net_type='vgg', version='0.1').cuda().eval()

    render_files = {p.name: p for p in Path(render_dir).iterdir() if p.is_file()}
    gt_files = {p.name: p for p in Path(gt_dir).iterdir() if p.is_file()}
    names = sorted(set(render_files) & set(gt_files))
    if not names:
        raise RuntimeError(f'No matching rendered/GT filenames in {render_dir} and {gt_dir}')
    if set(render_files) != set(gt_files):
        missing = sorted(set(gt_files) - set(render_files))
        extra = sorted(set(render_files) - set(gt_files))
        raise RuntimeError(f'Filename mismatch: missing renders={missing[:10]}, extra renders={extra[:10]}')
    values = {'PSNR': [], 'SSIM': [], 'LPIPS': []}
    with torch.no_grad():
        for name in names:
            render_tensor = TF.to_tensor(Image.open(render_files[name]).convert('RGB')).unsqueeze(0).cuda()
            gt_tensor = TF.to_tensor(Image.open(gt_files[name]).convert('RGB')).unsqueeze(0).cuda()
            values['PSNR'].append(float(psnr(render_tensor, gt_tensor).mean().item()))
            values['SSIM'].append(float(ssim(render_tensor, gt_tensor).mean().item()))
            if include_lpips:
                values['LPIPS'].append(float(lpips_model(render_tensor, gt_tensor).mean().item()))
            del render_tensor, gt_tensor
    result = {key: sum(items) / len(items) for key, items in values.items() if items}
    result['images'] = len(names)
    return result


metric_rows = []
if RUN_CHECKPOINT_METRICS:
    for scene in submission_scenes:
        model_path = OUTPUT_ROOT / scene['split'] / scene['scene_name']
        gt_dir = Path(scene['test_csv']).parent / 'images'
        if not gt_dir.exists():
            print(f"Skip {scene['split']}/{scene['scene_name']}: no public test ground truth at {gt_dir}")
            continue
        missing = [i for i in METRIC_ITERATIONS if not (model_path / 'point_cloud' / f'iteration_{i}' / 'point_cloud.ply').exists()]
        if missing:
            raise FileNotFoundError(f'{scene["scene_name"]}: missing point clouds for iterations {missing}')

        for iteration in METRIC_ITERATIONS:
            # Render the exact public test poses. Output names match image_name in the CSV.
            render_dir = model_path / 'checkpoint_eval' / f'iteration_{iteration}' / 'renders'
            if render_dir.exists():
                shutil.rmtree(render_dir)
            cmd = [
                'python', 'render_test_poses.py', '-m', str(model_path), '--test_csv', str(scene['test_csv']),
                '--out_dir', str(render_dir), '--iteration', str(iteration), '--output_name_mode', 'csv_exact',
                '--max_gaussian_scale', str(RENDER_MAX_GAUSSIAN_SCALE),
                '--max_render_points', str(RENDER_MAX_RENDER_POINTS),
                '--max_view_points', str(RENDER_MAX_VIEW_POINTS),
                '--view_cull_margin', str(RENDER_VIEW_CULL_MARGIN),
                '--min_view_depth', str(RENDER_MIN_VIEW_DEPTH),
                '--max_screen_radius_px', str(RENDER_MAX_SCREEN_RADIUS_PX),
                '--render_scale', str(RENDER_RESOLUTION_SCALE),
            ]
            if RENDER_USE_ANTIALIASING:
                cmd.append('--antialiasing')
            if RENDER_CONVERT_SHS_PYTHON:
                cmd.append('--convert_SHs_python')
            if RENDER_COMPUTE_COV3D_PYTHON:
                cmd.append('--compute_cov3D_python')
            run_stream(cmd, REPO_DIR, model_path / f'render_test_metrics_{iteration}.log')

            metrics = evaluate_render_folder(
                render_dir, gt_dir, include_lpips=METRIC_INCLUDE_LPIPS
            )
            row = {'split': scene['split'], 'scene': scene['scene_name'], 'iteration': iteration, **metrics}
            metric_rows.append(row)
            print(row)
            torch.cuda.empty_cache()

    metrics_df = pd.DataFrame(metric_rows).sort_values(['split', 'scene', 'iteration']).reset_index(drop=True)
    metric_columns = [c for c in ['PSNR', 'SSIM', 'LPIPS'] if c in metrics_df.columns]
    display(metrics_df)
    metrics_df.to_csv(WORK_DIR / 'checkpoint_metrics.csv', index=False)
    print('Saved:', WORK_DIR / 'checkpoint_metrics.csv')
    if len(METRIC_ITERATIONS) >= 2:
        first_iteration, last_iteration = METRIC_ITERATIONS[0], METRIC_ITERATIONS[-1]
        first = metrics_df[metrics_df.iteration == first_iteration].set_index(['split', 'scene'])
        last = metrics_df[metrics_df.iteration == last_iteration].set_index(['split', 'scene'])
        delta_df = (last[metric_columns] - first[metric_columns]).add_prefix(f'delta_{last_iteration}_minus_{first_iteration}_').reset_index()
        display(delta_df)
        delta_df.to_csv(WORK_DIR / 'checkpoint_metrics_delta.csv', index=False)
        print('Saved:', WORK_DIR / 'checkpoint_metrics_delta.csv')


## Visual comparison: ground truth and inference

This cell only displays images in the notebook; it does not create another output folder. Run the checkpoint-metric cell first so the inference images exist.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

VISUAL_COMPARE_ITERATION = 30000
VISUAL_COMPARE_COUNT = 5  # Use None to display every test image.

for scene in submission_scenes:
    gt_dir = Path(scene['test_csv']).parent / 'images'
    model_path = OUTPUT_ROOT / scene['split'] / scene['scene_name']
    render_dir = model_path / 'checkpoint_eval' / f'iteration_{VISUAL_COMPARE_ITERATION}' / 'renders'
    if not gt_dir.exists():
        print(f"Skip {scene['split']}/{scene['scene_name']}: no ground truth test images")
        continue
    if not render_dir.exists():
        print(f'Skip {scene["scene_name"]}: run the metric/render cell first; missing {render_dir}')
        continue

    with open(scene['test_csv'], newline='', encoding='utf-8-sig') as f:
        compare_rows = list(csv.DictReader(f))
    if VISUAL_COMPARE_COUNT is not None:
        compare_rows = compare_rows[:VISUAL_COMPARE_COUNT]

    print(f"{scene['split']}/{scene['scene_name']} - iteration {VISUAL_COMPARE_ITERATION}")
    for row in compare_rows:
        image_name = Path(row['image_name']).name
        gt_path = gt_dir / image_name
        inference_path = render_dir / image_name
        if not gt_path.exists() or not inference_path.exists():
            print(f'Skip {image_name}: GT or inference image is missing')
            continue

        with Image.open(gt_path) as gt_image, Image.open(inference_path) as inference_image:
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            axes[0].imshow(gt_image.convert('RGB'))
            axes[0].set_title('Ground truth')
            axes[1].imshow(inference_image.convert('RGB'))
            axes[1].set_title(f'Inference - iteration {VISUAL_COMPARE_ITERATION}')
            for axis in axes:
                axis.axis('off')
            fig.suptitle(image_name)
            plt.tight_layout()
            plt.show()


In [ ]:
# Export one downloadable PNG containing 6 Ground truth vs Inference comparisons.
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

COMPARISON_FILE = WORK_DIR / 'ground_truth_vs_inference.png'
MAX_COMPARISONS = 6
pairs = []
for scene in submission_scenes:
    gt_dir = Path(scene['test_csv']).parent / 'images'
    render_dir = OUTPUT_ROOT / scene['split'] / scene['scene_name'] / 'checkpoint_eval' / f'iteration_{ITERATIONS}' / 'renders'
    if not (gt_dir.exists() and render_dir.exists()):
        continue
    with open(scene['test_csv'], newline='', encoding='utf-8-sig') as f:
        for row in csv.DictReader(f):
            name = Path(row['image_name']).name
            gt_path, pred_path = gt_dir / name, render_dir / name
            if gt_path.exists() and pred_path.exists():
                pairs.append((scene['scene_name'], name, gt_path, pred_path))
                if len(pairs) >= MAX_COMPARISONS:
                    break
    if len(pairs) >= MAX_COMPARISONS:
        break
if not pairs:
    raise FileNotFoundError('No GT/inference pairs found. Run the checkpoint metric/render cell first.')

# Two comparison panels per row and three rows for six images.
columns, half_width, header = 2, 480, 52
with Image.open(pairs[0][2]) as sample:
    sample = ImageOps.exif_transpose(sample)
    image_height = max(1, round(half_width * sample.height / sample.width))
panel_width, panel_height = half_width * 2, header + image_height
rows = (len(pairs) + columns - 1) // columns
canvas = Image.new('RGB', (panel_width * columns, panel_height * rows), 'white')
draw = ImageDraw.Draw(canvas)

for index, (scene_name, image_name, gt_path, pred_path) in enumerate(pairs):
    x = (index % columns) * panel_width
    y = (index // columns) * panel_height
    with Image.open(gt_path) as a, Image.open(pred_path) as b:
        gt = ImageOps.fit(ImageOps.exif_transpose(a).convert('RGB'), (half_width, image_height), method=Image.Resampling.LANCZOS)
        pred = ImageOps.fit(ImageOps.exif_transpose(b).convert('RGB'), (half_width, image_height), method=Image.Resampling.LANCZOS)
    canvas.paste(gt, (x, y + header))
    canvas.paste(pred, (x + half_width, y + header))
    draw.text((x + 12, y + 7), f'{index + 1}. GROUND TRUTH', fill='black')
    draw.text((x + half_width + 12, y + 7), f'INFERENCE - ITERATION {ITERATIONS}', fill='black')
    draw.text((x + 12, y + 27), f'{scene_name} / {image_name}', fill='black')
    draw.line((x, y + panel_height - 1, x + panel_width, y + panel_height - 1), fill='#cccccc')

canvas.save(COMPARISON_FILE, 'PNG', optimize=True)
print(f'Saved {len(pairs)} comparisons to:', COMPARISON_FILE)
display(canvas)


In [ ]:
if not submission_scenes:
    print('No submission scenes configured; skip private render step.')
else:
    if SUBMISSION_DIR.exists():
        shutil.rmtree(SUBMISSION_DIR)
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

    for scene in submission_scenes:
        key = (scene['split'], scene['scene_name'])
        model_path = OUTPUT_ROOT / scene['split'] / scene['scene_name']
        out_dir = SUBMISSION_DIR / scene['scene_name']
        cmd = [
            'python', 'render_test_poses.py', '-m', str(model_path), '--test_csv', str(scene['test_csv']),
            '--out_dir', str(out_dir), '--iteration', str(ITERATIONS), '--output_name_mode', SUBMISSION_IMAGE_NAME_MODE,
            '--max_gaussian_scale', str(RENDER_MAX_GAUSSIAN_SCALE),
            '--max_render_points', str(RENDER_MAX_RENDER_POINTS),
            '--max_view_points', str(RENDER_MAX_VIEW_POINTS),
            '--view_cull_margin', str(RENDER_VIEW_CULL_MARGIN),
            '--min_view_depth', str(RENDER_MIN_VIEW_DEPTH),
            '--max_screen_radius_px', str(RENDER_MAX_SCREEN_RADIUS_PX),
            '--render_scale', str(RENDER_RESOLUTION_SCALE),
        ]
        if RENDER_USE_ANTIALIASING:
            cmd.append('--antialiasing')
        if RENDER_CONVERT_SHS_PYTHON:
            cmd.append('--convert_SHs_python')
        if RENDER_COMPUTE_COV3D_PYTHON:
            cmd.append('--compute_cov3D_python')
        run_stream(cmd, REPO_DIR, model_path / 'render_test_poses.log')

    print('Rendered submission images to:', SUBMISSION_DIR)

In [ ]:
if not submission_scenes:
    print('No submission scenes configured; skip submission validation/zip step.')
else:
    from PIL import Image

    def expected_output_name(image_name):
        p = Path(image_name).name
        if SUBMISSION_IMAGE_NAME_MODE == 'stem_png':
            return Path(p).stem + '.png'
        return p

    errors = []
    total_expected = 0
    expected_scene_names = {scene['scene_name'] for scene in submission_scenes}
    actual_scene_names = {p.name for p in SUBMISSION_DIR.iterdir() if p.is_dir()}
    if actual_scene_names != expected_scene_names:
        errors.append(f'Scene folder mismatch: got {sorted(actual_scene_names)}, expected {sorted(expected_scene_names)}')
    for scene in submission_scenes:
        scene_out = SUBMISSION_DIR / scene['scene_name']
        with open(scene['test_csv'], newline='', encoding='utf-8-sig') as f:
            rows = list(csv.DictReader(f))
        expected_files = {expected_output_name(row['image_name']) for row in rows}
        actual_files = {p.name for p in scene_out.iterdir() if p.is_file()} if scene_out.exists() else set()
        if actual_files != expected_files:
            missing = sorted(expected_files - actual_files)
            extra = sorted(actual_files - expected_files)
            if missing:
                errors.append(f"{scene['scene_name']}: missing {len(missing)} files, e.g. {missing[:10]}")
            if extra:
                errors.append(f"{scene['scene_name']}: extra {len(extra)} files, e.g. {extra[:10]}")
        total_expected += len(rows)
        for row in rows:
            out_path = scene_out / expected_output_name(row['image_name'])
            if not out_path.exists():
                errors.append(f'Missing: {out_path}')
                continue
            with Image.open(out_path) as im:
                expected_size = (int(float(row['width'])), int(float(row['height'])))
                if im.size != expected_size:
                    errors.append(f'Size mismatch: {out_path} got {im.size}, expected {expected_size}')

    print('Expected images:', total_expected)
    if errors:
        print('\n'.join(errors[:50]))
        raise RuntimeError(f'Submission validation failed with {len(errors)} errors')
    print('Submission validation OK')

    if SUBMISSION_ZIP.exists():
        SUBMISSION_ZIP.unlink()
    archive_path = shutil.make_archive(str(SUBMISSION_ZIP.with_suffix('')), 'zip', SUBMISSION_DIR)
    print('Created:', archive_path)



In [ ]:
from IPython.display import FileLink

# Click vao link ben duoi de tai truc tiep file submission.zip ve may
FileLink('submission.zip')